In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve()
sys.path.append(str(ROOT / "src"))

print("Project root:", ROOT)

Project root: /Users/sydverma/Desktop/AllerScan


In [2]:
import cv2
import easyocr
import re

image_path = "input3.jpg"

image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError(f"Could not find image: {image_path}")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

reader = easyocr.Reader(["en"])
results = reader.readtext(blurred)

def clean_text(text):
    text = re.sub(r"\$?\d+(\.\d+)?", "", text)
    text = re.sub(r"[^a-zA-Z& ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

JUNK_WORDS = [
    "DELICIOUS FOOD",
    "MENU",
    "APPETIZER",
    "APPETIZERS",
    "SALAD & SOUP",
    "SALAD SOUP",
    "DRINKS",
    "D RINKS",
    "FOOD",
]

foodItems = []

for _, text, conf in results:
    if conf > 0.5:
        cleaned = clean_text(text)

        if len(cleaned) > 2 and cleaned.upper() not in JUNK_WORDS:
            foodItems.append(cleaned)

print("Detected food items:")
for item in foodItems:
    print("-", item)

/Users/sydverma/lemonades_group_project/.venv314/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Detected food items:
- Mushroom Burger
- Crispy Fried Chicken
- Fish & Chips
- Spaghetti & Meatballs
- Hotdog Sandwich
- Macaroni Soup
- Tex Mex Chili
- French Fries
- Calamari
- Beef Taco
- Purified Water
- Sparkling Water
- Soda In A Bottle
- Orange Juice
- Fresh Lemonade


In [3]:
from predict_allergens import predict_allergens

results = predict_allergens(foodItems)

print("--- Allergen Detection Results ---\n")

for r in results:
    print(r["food_item"])
    print("  Allergens:", ", ".join(r["predicted_allergens"]))
    print("  Matched:", ", ".join(r["matched_ingredients"]))
    print("  Method:", r["method"])
    print("  Confidence:", f"{r['confidence']:.1%}")
    print()

--- Allergen Detection Results ---

Mushroom Burger
  Allergens: Dairy, Wheat
  Matched: bread, burger, cheese, wheat
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Crispy Fried Chicken
  Allergens: Eggs, Wheat
  Matched: batter, breaded, egg, wheat
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Fish & Chips
  Allergens: Fish
  Matched: fish
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Spaghetti & Meatballs
  Allergens: Wheat
  Matched: pasta, spaghetti, wheat
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Hotdog Sandwich
  Allergens: Wheat
  Matched: bread, sandwich, wheat
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Macaroni Soup
  Allergens: Celery, Dairy, Garlic, Onion, Wheat
  Matched: celery, cheese, garlic, macaroni, onion, pasta, wheat
  Method: rule-based + ingredient hints
  Confidence: 95.0%

Tex Mex Chili
  Allergens: Garlic, Onion
  Matched: garlic, onion
  Method: rule-based + ingredient hints
  Con

In [4]:
import pandas as pd

rows = []

for r in results:
    rows.append({
        "food_item": r["food_item"],
        "allergens": ", ".join(r["predicted_allergens"]),
        "matched_ingredients": ", ".join(r["matched_ingredients"]),
        "method": r["method"],
        "confidence": r["confidence"]
    })

df = pd.DataFrame(rows)

output_path = "allergen_results.csv"
df.to_csv(output_path, index=False)

print(f"Saved results to {output_path}")
df.head()

Saved results to allergen_results.csv


,food_item,allergens,matched_ingredients,method,confidence
0,Mushroom Burger,"Dairy, Wheat","bread, burger, cheese, wheat",rule-based + ingredient hints,0.95
1,Crispy Fried Chicken,"Eggs, Wheat","batter, breaded, egg, wheat",rule-based + ingredient hints,0.95
2,Fish & Chips,Fish,fish,rule-based + ingredient hints,0.95
3,Spaghetti & Meatballs,Wheat,"pasta, spaghetti, wheat",rule-based + ingredient hints,0.95
4,Hotdog Sandwich,Wheat,"bread, sandwich, wheat",rule-based + ingredient hints,0.95
